# ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೆ임್‌ವರ್ಕ್ — ಅಜ್ಯೂರ್ ಓಪನ್‌ಎಐ (ಪ್ರತಿಕ್ರಿಯೆಗಳ API)

ಈ ಕೋಡ್ ಮಾದರಿಯಲ್ಲಿ, ನೀವು **ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೆ임್‌ವರ್ಕ್ (MAF)** ಅನ್ನು ಬಳಸಿಕೊಂಡು **ಅಜ್ಯೂರ್ ಓಪನ್‌ಎಐ** ಪ್ರಧಾನವಾಗಿ ಬೆಂಬಲಿತ ಒಂದು ಸರಳ ಏಜೆಂಟ್ ಅನ್ನು **ಪ್ರತಿಕ್ರಿಯೆಗಳ API** ಮೂಲಕ ರಚಿಸುವಿರಿ.

> **ಸ್ಥಳಾಂತರ ಸೂಚನೆ:** ಈ ಮಾದರಿ ಹಿಂದಿನದಾಗಿ Semantic Kernel ಮತ್ತು GitHub ಮಾದರಿಗಳನ್ನು ಬಳಸಿಕೊಂಡಿತ್ತು. ಇದನ್ನು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೆ임್‌ವರ್ಕ್‌ಗೆ ಸ್ಥಳಾಂತರಿಸಲಾಗಿದೆ, ಮತ್ತು GitHub ಮಾದರಿಗಳನ್ನು (ಪ್ರಚಲಿತ ಇಲ್ಲದವು, 2026 ಜುಲೈಗೆ ನಿವೃತ್ತಿಯಾಗಲಿವೆ) ಬದಲಾಗಿ ಅಜ್ಯೂರ್ ಓಪನ್‌ಎಐ, ಪ್ರತಿಕ್ರಿಯೆಗಳ APIನ್ನು ಬೆಂಬಲಿಸುವದಾಗಿ ಪರಿವರ್ತಿಸಲಾಗಿದೆ. MAFಯಲ್ಲಿನ `OpenAIChatClient` ಅಜ್ಯೂರ್ ಓಪನ್‌ಎಐಯ ಸ್ಥಿರವಾದ `/openai/v1/` ಎಂಡ್‌ಪಾಯಿಂಟ್ ಅನ್ನು ಗುರಿಯಾಗಿದೆ ಮತ್ತು ಡಿಫಾಲ್ಟ್ ಆಗಿ ಪ್ರತಿಕ್ರಿಯೆಗಳ APIನ್ನು ಉಪಯೋಗಿಸುತ್ತದೆ.

ಈ ಮಾದರಿಯ ಉದ್ದೇಶ ಮುಂದಿನ ಕೋಡ್ ಮಾದರಿಗಳಲ್ಲಿ ವಿವಿಧ ಏಜೆಂಟ್ ಮಾದರಿಗಳನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸುವಾಗ ಕ್ರಮಗಳನ್ನು ತೋರಿಸುವುದು.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## ಅಗತ್ಯವಿರುವ ಪೈಥಾನ್ ಪ್ಯಾಕೇಜ್ಗಳನ್ನು ಆಮದುಮಾಡಿ


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ಉಪಕರಣವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು

ಮೈಕ್ರೋಸಾಫ್ಟ್ಅಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್‌ನಲ್ಲಿ, **ಉಪಕರಣ** ಎಂದರೆ `@tool` ಮೂಲಕ ಅಲಂಕರಿಸಲಾದ ಸರಳ Python ಕಾರ್ಯವಾಗಿದ್ದು, ಅದನ್ನು ಏಜೆಂಟ್ ಕರೆಮಾಡಬಹುದು. ಕೆಳಗೆ ನಾವು ಯಾದೃಚ್ಛಿಕ ರಜಾ ಗಮ್ಯಸ್ಥಳವನ್ನು ಹಿಂತಿರುಗಿಸುವ ಉಪಕರಣವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತೇವೆ ಮತ್ತು ಹಿಂದಿನದ್ದನ್ನು ಪುನರಾವರ್ತಿಸುವುದನ್ನು ತಪ್ಪಿಸುತ್ತೇವೆ.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ಏಜೆಂಟ್ ರಚನೆ

ಇಲ್ಲಿ, `TravelAgent` ಎಂಬ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ.

ಈ ಉದಾಹರಣೆಯಲ್ಲಿ, ನಾವು ಅತ್ಯಂತ ಮೂಲಭೂತ ಸೂಚನೆಗಳನ್ನು ಬಳಸುತ್ತೇವೆ. ಏಜೆಂಟ್‌ನ ವರ್ತನೆ ಹೇಗೆ ಬದಲಾಗುತ್ತದೆ ಎಂದು ಗಮನಿಸಲು ಈ ಸೂಚನೆಗಳನ್ನು ನಿಮಗೆ ಇಚ್ಛೆಯಂತೆ ಬದಲಾಯಿಸಬಹುದು.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ಏಜೆಂಟ್ ಅನ್ನು ಚಾಲನೆ ಮಾಡುವುದು

ಈಗ ನಾವು ಏಜೆಂಟ್ ಅನ್ನು ಚಾಲನೆ ಮಾಡಬಹುದು. ನಾವು `AgentSession` ಅನ್ನು ರಚಿಸುತ್ತೇವೆ ಹಾಗಾಗಿ ಏಜೆಂಟ್ ಸಂವಾದವನ್ನು ಟರ್ನ್‌ಗಳ ಮೂಲಕ ಸ್ಮರಿಸುತ್ತದೆ, ನಂತರ ಎರಡು `user_inputs` ಅನ್ನು ಕಳುಹಿಸುತ್ತೇವೆ. ಮೊದಲದವು ಒಂದು ಪ್ರವಾಸವನ್ನು ಕೇಳುತ್ತದೆ; ಎರಡನೆಯದು ಬಳಕೆದಾರರಿಗೆ ಆ ಸಲಹೆ ಇಷ್ಟವಾಗಲಿಲ್ಲವೆಂಬುದನ್ನು ಹೇಳುತ್ತದೆ ಮತ್ತು ಮತ್ತೊಂದುನ್ನು ಕೇಳುತ್ತದೆ — ಏಜೆಂಟ್ ಸೆಷನ್ ಇತಿಹಾಸವನ್ನು ಮತ್ತು `get_random_destination` ಉಪಕರಣವನ್ನು ಬಳಸಿ ಪ್ರತಿಕ್ರಿಯಿಸುತ್ತದೆ.

ನೀವು ಈ ಸಂದೇಶಗಳನ್ನು ಬದಲಾಯಿಸಿ ಏಜೆಂಟ್ ಹೇಗೆ ವಿಭಿನ್ನವಾಗಿ ಪ್ರತಿಕ್ರಿಯಿಸುತ್ತದೆ ಎಂಬುದನ್ನು ವೀಕ್ಷಿಸಬಹುದು. ಪ್ರತಿಕ್ರಿಯೆಗಳು **ಸ್ಟ್ರೀಮ್** ಆಗಿ ಟೋಕನ್-ಬೈ-ಟೋಕನ್ ರೂಪದಲ್ಲಿ ಬರುತ್ತವೆ.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
